In [ ]:
"""
4. Movement Pruning — ResNet-50 / CIFAR-10
==========================================
TRUE movement pruning (Sanh et al., 2020).

Core idea:
  Weights that move TOWARD zero during fine-tuning are pruned.
  Weights that move AWAY from zero are kept — even if currently small.

  Score(w_i) = w_i(t_final) - w_i(t_0)     [weight displacement]
             = sign(w_i(t_final)) × |movement|

  Concretely:
    - Positive weight that grew   → large positive score  → KEEP
    - Positive weight that shrank → small/negative score  → PRUNE
    - Negative weight that grew   → small/negative score  → PRUNE
    - Negative weight that became more negative → KEEP

This requires an actual fine-tuning phase (not just a forward pass).
We fine-tune the pre-trained model for a few epochs, then score by movement.

Output: 4_movement_pruning.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import (precision_score, recall_score,
                             f1_score, accuracy_score)

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "4_movement_pruning_new.json"

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
NUM_WORKERS = 2
NUM_CLASSES = 10

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

# ── Movement pruning hyperparameters ──────────────────────────────────────────
# Fine-tuning is REQUIRED for true movement pruning.
# We fine-tune briefly (few epochs) at a low LR so the model doesn't drift far
# from the pre-trained checkpoint — we just want weights to "reveal their intent".
FINETUNE_EPOCHS   = 3        # how many epochs to fine-tune before scoring
FINETUNE_LR       = 1e-4     # low LR: small controlled movements
FINETUNE_MOMENTUM = 0.9
FINETUNE_WD       = 1e-4

SPARSITY_LEVELS = [0.30, 0.50, 0.70, 0.80, 0.90]

MAX_ACC_DROP  = 0.02
INFERENCE_RUNS = 100


# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes=10):
    """Same architecture as baseline — MUST match exactly."""
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path):
    model = build_model(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model


# ── Data ──────────────────────────────────────────────────────────────────────
def get_loaders():
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    train_ds = torchvision.datasets.CIFAR10(root="../data", train=True,
                                             download=True, transform=transform_train)
    test_ds  = torchvision.datasets.CIFAR10(root="../data", train=False,
                                             download=True, transform=transform_test)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, test_loader


# ── Fine-tuning phase ─────────────────────────────────────────────────────────
def finetune(model, train_loader, n_epochs, lr):
    """
    Brief fine-tuning to let weights reveal their movement direction.

    Key points:
    - We use a LOW learning rate to keep movements small and controlled.
    - We do NOT use label smoothing here — clean gradients matter more.
    - The goal is not to improve accuracy, but to surface which weights
      the optimizer naturally pushes toward zero vs. away from zero.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr,
                                momentum=FINETUNE_MOMENTUM,
                                weight_decay=FINETUNE_WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    model.train()
    for epoch in range(n_epochs):
        total_loss, correct, total = 0.0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss    = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct    += outputs.argmax(1).eq(targets).sum().item()
            total      += targets.size(0)
        scheduler.step()
        print(f"    Fine-tune epoch {epoch+1}/{n_epochs} | "
              f"Loss: {total_loss/len(train_loader):.4f} | "
              f"Acc: {correct/total:.4f}")
    return model


# ── Movement score computation ─────────────────────────────────────────────────
def compute_movement_scores(w_before: dict, w_after: dict) -> dict:
    """
    True movement score (Sanh et al. 2020, Eq. 2):

        Score(w_i) = w_i(after) - w_i(before)

    Interpretation:
      - A weight starting at +0.5 that moved to +0.8 → score = +0.3 → KEEP
        (moved away from zero)
      - A weight starting at +0.5 that moved to +0.1 → score = -0.4 → PRUNE
        (moved toward zero)
      - A weight starting at -0.3 that moved to -0.6 → score = -0.3 → KEEP
        (became more negative = moved away from zero; the sign of the original
         weight matters, so we use signed movement)

    To handle negative weights correctly, we follow the paper's formulation:
        Score(w_i) = sign(w_i(before)) × (w_i(after) - w_i(before))
                   = sign(w_i) × Δw_i

    Positive score → weight moved in the direction it already pointed → KEEP
    Negative score → weight moved toward zero → PRUNE
    """
    scores = {}
    for name in w_before:
        delta  = w_after[name] - w_before[name]          # Δw = w_final - w_init
        sign   = torch.sign(w_before[name])               # sign of initial weight
        # Weights exactly at zero get score=0 (neither kept nor specifically pruned;
        # they'll be captured by the threshold naturally)
        scores[name] = sign * delta
    return scores


def apply_movement_pruning(model, train_loader, sparsity):
    """
    Full movement pruning pipeline:
      1. Snapshot weights before fine-tuning  (w_before)
      2. Fine-tune the model briefly
      3. Snapshot weights after fine-tuning   (w_after)
      4. Compute movement scores
      5. Find global threshold at target sparsity
      6. Zero out weights with the LOWEST scores (moved toward zero)
    """
    pruned = copy.deepcopy(model)

    # ── Step 1: snapshot initial weights ──────────────────────────────────────
    print("    Snapshotting initial weights...")
    w_before = {}
    for name, module in pruned.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            w_before[name] = module.weight.data.clone().cpu()

    # ── Step 2: fine-tune ─────────────────────────────────────────────────────
    print(f"    Fine-tuning for {FINETUNE_EPOCHS} epoch(s) at LR={FINETUNE_LR}...")
    pruned = finetune(pruned, train_loader, FINETUNE_EPOCHS, FINETUNE_LR)

    # ── Step 3: snapshot post-fine-tune weights ────────────────────────────────
    w_after = {}
    for name, module in pruned.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            w_after[name] = module.weight.data.clone().cpu()

    # ── Step 4: compute movement scores ───────────────────────────────────────
    print("    Computing movement scores...")
    scores = compute_movement_scores(w_before, w_after)

    # ── Step 5: global threshold ───────────────────────────────────────────────
    all_scores = torch.cat([s.view(-1) for s in scores.values()])
    k          = max(1, int(all_scores.numel() * sparsity))
    threshold  = torch.kthvalue(all_scores, k).values.item()
    print(f"    Score threshold at {sparsity*100:.0f}% sparsity: {threshold:.6f}")

    # ── Step 6: apply binary mask ──────────────────────────────────────────────
    # Prune weights whose movement score ≤ threshold (moved toward zero or stayed)
    with torch.no_grad():
        for name, module in pruned.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)) and name in scores:
                score_tensor = scores[name].to(DEVICE)
                mask = (score_tensor > threshold).float()
                module.weight.data *= mask

    pruned.eval()
    return pruned, scores, w_before, w_after


# ── Utility ────────────────────────────────────────────────────────────────────
def real_sparsity(model):
    zeros = total = 0
    for _, m in model.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            zeros += (m.weight == 0).sum().item()
            total += m.weight.numel()
    return zeros / total if total else 0.0

def count_params(model):
    total  = sum(p.numel() for p in model.parameters())
    active = sum((p != 0).sum().item() for p in model.parameters())
    return total, active

def disk_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / 1024**2
    finally:
        os.remove(tmp)
    return size

def sparse_size_mb(model):
    active = sum((p != 0).sum().item() for p in model.parameters())
    return active * 4 / 1024**2   # float32 = 4 bytes

def movement_stats(w_before, w_after):
    """Summary statistics about how weights moved during fine-tuning."""
    all_delta = []
    for name in w_before:
        delta = (w_after[name] - w_before[name]).view(-1)
        all_delta.append(delta)
    all_delta = torch.cat(all_delta)
    return {
        "mean_abs_movement" : float(all_delta.abs().mean()),
        "max_abs_movement"  : float(all_delta.abs().max()),
        "std_movement"      : float(all_delta.std()),
        "pct_moved_toward_zero": float(
            # fraction of weights whose signed movement was negative
            (all_delta < 0).float().mean()
        ),
    }


# ── Evaluation ────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def measure_gpu_ms(model):
    if DEVICE.type != "cuda": return None
    model.eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32, device=DEVICE)
    with torch.no_grad():
        for _ in range(20): model(dummy)
        torch.cuda.synchronize()
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(INFERENCE_RUNS): model(dummy)
        e.record()
        torch.cuda.synchronize()
    return s.elapsed_time(e) / INFERENCE_RUNS

def measure_cpu_ms(model):
    cpu_m = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32)
    with torch.no_grad():
        for _ in range(5): cpu_m(dummy)
        t0 = time.perf_counter()
        for _ in range(20): cpu_m(dummy)
    return (time.perf_counter() - t0) / 20 * 1000


from thop import profile

def compute_flops(model, device, input_size=(1, 3, 32, 32)):
    """
    Standard FLOPs computation using THOP.
    Returns:
        flops_total (int): total FLOPs (not MACs)
        flops_G (float): FLOPs in GFLOPs
    """
    model.eval()
    dummy = torch.randn(input_size).to(device)

    macs, _ = profile(model, inputs=(dummy,), verbose=False)
    flops = macs * 2  # convert MACs → FLOPs

    return int(flops), flops / 1e9


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print("  4. Movement Pruning — ResNet-50 / CIFAR-10")
    print("  Method: Sanh et al. (2020) — sign(w₀) × Δw")
    print(f"  Device: {DEVICE}")
    print(f"{'='*65}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    print(f"  Baseline accuracy: {baseline['accuracy']:.4f}\n")

    model = load_baseline(BASELINE_MODEL_PATH)
    train_loader, test_loader = get_loaders()

    base_disk            = disk_size_mb(model)
    base_flops, base_flops_G = compute_flops(model, DEVICE)

    results    = []
    best_model = None
    best_score = -1.0
    best_sp    = None

    for sparsity in SPARSITY_LEVELS:
        print(f"\n{'─'*55}")
        print(f"  Target sparsity: {int(sparsity*100)}%")
        print(f"{'─'*55}")

        pruned, scores, w_before, w_after = apply_movement_pruning(
            model, train_loader, sparsity
        )

        actual_sp         = real_sparsity(pruned)
        total_p, active_p = count_params(pruned)
        metrics           = evaluate(pruned, test_loader)
        inf_gpu           = measure_gpu_ms(pruned)
        inf_cpu           = measure_cpu_ms(pruned)
        acc_drop          = baseline["accuracy"] - metrics["accuracy"]
        sp_mb             = sparse_size_mb(pruned)
        dk_mb             = disk_size_mb(pruned)
        mv_stats          = movement_stats(w_before, w_after)

        pruned_flops, pruned_flops_G = compute_flops(pruned, DEVICE)

        row = {
            "target_sparsity"             : sparsity,
            "actual_sparsity"             : round(actual_sp, 4),
            "accuracy"                    : round(metrics["accuracy"], 6),
            "precision"                   : round(metrics["precision"], 6),
            "recall"                      : round(metrics["recall"], 6),
            "f1"                          : round(metrics["f1"], 6),
            "accuracy_drop"               : round(acc_drop, 6),
            "within_threshold"            : bool(acc_drop <= MAX_ACC_DROP),
            "params_total"                : total_p,
            "params_active"               : active_p,
            "size_sparse_theoretical_mb"  : round(sp_mb, 4),
            "size_disk_mb"                : round(dk_mb, 4),
            "disk_saved_mb"               : round(base_disk - dk_mb, 4),
            "inference_gpu_ms"            : round(inf_gpu, 4) if inf_gpu else None,
            "inference_cpu_ms"            : round(inf_cpu, 4),
            "flops_total"                 : int(base_flops),
            "flops_active"  : pruned_flops,
            "flops_reduction_pct": round((1 - pruned_flops / base_flops) * 100, 2),
            "finetune_epochs"             : FINETUNE_EPOCHS,
            "finetune_lr"                 : FINETUNE_LR,
            "movement_stats"              : {k: round(v, 6) for k, v in mv_stats.items()},
        }
        results.append(row)

        print(f"\n  ✓ Actual sparsity : {actual_sp*100:.1f}%")
        print(f"  ✓ Accuracy        : {metrics['accuracy']:.4f}  (drop: {acc_drop:+.4f})")
        print(f"  ✓ Disk size       : {dk_mb:.1f} MB  (saved: {base_disk - dk_mb:.1f} MB)")
        print(f"  ✓ FLOPs reduction : {row['flops_reduction_pct']:.1f}%")
        print(f"  ✓ Mean |Δw|       : {mv_stats['mean_abs_movement']:.6f}")
        print(f"  ✓ % moved → zero  : {mv_stats['pct_moved_toward_zero']*100:.1f}%")

        if acc_drop <= MAX_ACC_DROP:
            score = metrics["accuracy"] + ((base_disk - dk_mb) / base_disk) * 0.1
            if score > best_score:
                best_score = score
                best_model = copy.deepcopy(pruned)
                best_sp    = sparsity

    # ── Save best model ────────────────────────────────────────────────────────
    if best_model is not None:
        torch.save(best_model.state_dict(), "4_movement_pruned_best.pth")
        print(f"\n  ✓ Best model saved (sparsity={int(best_sp*100)}%) "
              f"→ 4_movement_pruned_best.pth")

    # ── Save report ────────────────────────────────────────────────────────────
    report = {
        "method"               : "movement_pruning",
        "description"          : (
            "True movement pruning (Sanh et al. 2020). "
            "Scores = sign(w_before) × (w_after − w_before). "
            "Weights moving toward zero are pruned."
        ),
        "pruning_granularity"  : "weight (unstructured)",
        "scoring_formula"      : "sign(w_i(t0)) × (w_i(t_final) − w_i(t0))",
        "reference"            : "Sanh et al., 'Movement Pruning: Adaptive Sparsity by Fine-Tuning', NeurIPS 2020",
        "finetune_config"      : {
            "epochs"   : FINETUNE_EPOCHS,
            "lr"       : FINETUNE_LR,
            "momentum" : FINETUNE_MOMENTUM,
            "weight_decay": FINETUNE_WD,
        },
        "baseline"             : baseline,
        "device"               : str(DEVICE),
        "max_acc_drop_threshold": MAX_ACC_DROP,
        "best_sparsity"        : best_sp,
        "results"              : results,
        "implementation_notes" : {
            "why_finetune_needed" : (
                "Movement scores require observing HOW weights change during "
                "gradient descent. Without a training loop, weights cannot move, "
                "so movement = 0 everywhere — pruning degenerates to random."
            ),
            "score_sign_convention" : (
                "sign(w_before) × Δw is positive when a weight moves AWAY from "
                "zero (regardless of its sign), and negative when it moves toward "
                "zero. We prune the lowest-scoring weights globally."
            ),
            "vs_magnitude_pruning" : (
                "A large weight that shrinks toward zero during fine-tuning will "
                "be pruned by movement pruning but kept by magnitude pruning. "
                "A small weight that grows will be kept by movement pruning but "
                "pruned by magnitude pruning. This is the key behavioral difference."
            ),
            "vs_taylor_pruning" : (
                "Taylor pruning (|w × grad|) estimates loss sensitivity at a "
                "single point. Movement pruning integrates gradient signals over "
                "the entire fine-tuning trajectory, making it more robust."
            ),
            "limitation" : (
                "One-shot movement pruning (score then prune once) is used here. "
                "The full Sanh et al. method trains continuous soft mask scores "
                "jointly with weights using a straight-through estimator, enabling "
                "iterative pruning during training."
            ),
        },
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  ✓ Report saved → {OUTPUT_JSON}")
    print(f"\n{'='*65}")
    print("  Movement Pruning complete.")
    print(f"{'='*65}\n")


if __name__ == "__main__":
    main()


  4. Movement Pruning — ResNet-50 / CIFAR-10
  Method: Sanh et al. (2020) — sign(w₀) × Δw
  Device: cuda

  Baseline accuracy: 0.9320


───────────────────────────────────────────────────────
  Target sparsity: 30%
───────────────────────────────────────────────────────
    Snapshotting initial weights...
    Fine-tuning for 3 epoch(s) at LR=0.0001...
    Fine-tune epoch 1/3 | Loss: 0.1108 | Acc: 0.9889
    Fine-tune epoch 2/3 | Loss: 0.0948 | Acc: 0.9884
    Fine-tune epoch 3/3 | Loss: 0.0901 | Acc: 0.9881
    Computing movement scores...
    Score threshold at 30% sparsity: -0.000000

  ✓ Actual sparsity : 30.0%
  ✓ Accuracy        : 0.1000  (drop: +0.8320)
  ✓ Disk size       : 90.0 MB  (saved: -0.0 MB)
  ✓ FLOPs reduction : 0.0%
  ✓ Mean |Δw|       : 0.000009
  ✓ % moved → zero  : 50.9%

───────────────────────────────────────────────────────
  Target sparsity: 50%
───────────────────────────────────────────────────────
    Snapshotting initial weights...
    Fine-tuning for 3 epo